In [3]:
from PIL import Image
import numpy as np
from opensimplex import OpenSimplex

filename = 'general_images/1024x1024RGB_image_sample.png'
# create filename2, which adds the suffix "d" to the filename
filename2save = filename[:-4] + '_Dist' + filename[-4:]


# Load image, convert to numpy array
img = Image.open(filename)
img_data = np.array(img)
#get h and w of image
h, w, _ = img_data.shape # ignore the 3rd dimension (RGB)

# Get last 4 chars of filename
seed_from_filename = int(np.random.randint(0, 10000, 1)[0])
# Create a new array to hold the distorted image data
distorted_data = np.empty_like(img_data)

# Distort image with Perlin noise
distort_intensity = 200
distort_scale = 500.0 # higher is smoother

# Create arrays to hold the distortions
x_distort = np.empty((h,w), dtype=np.float64)
y_distort = np.empty((h,w), dtype=np.float64)

# Create a noise generator
gen = OpenSimplex(seed=seed_from_filename)

# Generate Perlin noise and store the distortions
for i in range(h):
    for j in range(w):
        # Generate Perlin noise based on the pixel coordinates and the seed
        n = gen.noise2(i / distort_scale, j / distort_scale)
        
        # Map the noise to [0, 1]
        n = (n + 1) / 2
        
        # Store the distortions
        x_distort[i, j] = n * distort_intensity
        y_distort[i, j] = n * distort_intensity

# Calculate the mean distortions
x_mean_distort = np.mean(x_distort)
y_mean_distort = np.mean(y_distort)

# Apply the distortions and translate the image back to its original position
for i in range(h):
    for j in range(w):
        # Use the Perlin noise to distort the pixel coordinates
        x = int((i + x_distort[i, j] - x_mean_distort) % h)
        y = int((j + y_distort[i, j] - y_mean_distort) % w)
        
        # Copy the pixel data
        distorted_data[i, j] = img_data[x, y]

# Create a new image from the distorted data
distorted_img = Image.fromarray(distorted_data)

distorted_img.show()
#wait for user to close image
input("Press Enter to continue...")

distorted_img.save(filename2save)